In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("Updated_dynamic-pricing-dataset.csv")
print(df.shape) 
print(df.head())

(73100, 17)
         Date Store ID Product ID     Category Region  Inventory Level  \
0  01-01-2022     S001      P0001    Groceries  North              231   
1  01-01-2022     S001      P0002         Toys  South              204   
2  01-01-2022     S001      P0003         Toys   West              102   
3  01-01-2022     S001      P0004         Toys  North              469   
4  01-01-2022     S001      P0005  Electronics   East              166   

   Units Sold  Units Ordered  Demand Forecast  Price  Discount  \
0         127             55           135.47  33.50        20   
1         150             66           144.04  63.01        20   
2          65             51            74.02  27.99        10   
3          61            164            62.18  32.72        10   
4          14            135             9.26  73.64         0   

  Weather Condition  Holiday/Promotion  Competitor Pricing Seasonality  \
0             Rainy                  0               29.69      Autumn  

In [4]:
df.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition      object
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality            object
Visitors                int64
Cost                  float64
dtype: object

In [6]:
# Convert the Date column from object (string) to datetime format
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

df.dtypes

Date                  datetime64[ns]
Store ID                      object
Product ID                    object
Category                      object
Region                        object
Inventory Level                int64
Units Sold                     int64
Units Ordered                  int64
Demand Forecast              float64
Price                        float64
Discount                       int64
Weather Condition             object
Holiday/Promotion              int64
Competitor Pricing           float64
Seasonality                   object
Visitors                       int64
Cost                         float64
dtype: object

In [7]:
df.isnull().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forecast       0
Price                 0
Discount              0
Weather Condition     0
Holiday/Promotion     0
Competitor Pricing    0
Seasonality           0
Visitors              0
Cost                  0
dtype: int64

In [9]:
# Check how many duplicate rows exist in the dataset
df.duplicated().sum()

# Remove duplicate rows if any exist
df.drop_duplicates(inplace=True)

# Check dataset shape again after removing duplicates
print(df.shape)

(73100, 17)


In [11]:
df.columns = df.columns.str.replace("/", "_")

print(df.columns)

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday_promotion',
       'competitor_pricing', 'seasonality', 'visitors', 'cost'],
      dtype='object')


In [12]:
# Check if any inventory values are negative
df[df["inventory_level"] < 0]

# Remove rows where inventory_level is negative
df = df[df["inventory_level"] >= 0]

In [13]:
# Check for rows where units_sold is greater than inventory_level
df[df["units_sold"] > df["inventory_level"]]

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,visitors,cost


In [14]:
# Check for rows where price is zero or negative
df[df["price"] <= 0]

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,visitors,cost


In [15]:
# Check rows where cost is greater than price
df[df["cost"] > df["price"]]

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,visitors,cost


In [18]:
# Check for invalid discount values
# Discount should always be between 0 and 1
df[(df["discount"] < 0) | (df["discount"] > 1)]

# Fix discount values by clipping them between 0 and 1
df["discount"] = df["discount"].clip(0, 1)

In [19]:
df[df["visitors"] < 0]

# Remove rows where visitors are negative
df = df[df["visitors"] >= 0]

In [20]:
df["category"] = df["category"].str.strip().str.title()

# Check unique category values
df["category"].unique()

array(['Groceries', 'Toys', 'Electronics', 'Furniture', 'Clothing'],
      dtype=object)

In [21]:
df["weather_condition"] = df["weather_condition"].str.title()

df["weather_condition"].unique()

array(['Rainy', 'Sunny', 'Cloudy', 'Snowy'], dtype=object)

In [22]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)

# Calculate the Interquartile Range
IQR = Q3 - Q1

# Define lower and upper bounds for normal price values
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Remove rows where price is outside the normal range
df = df[(df["price"] >= lower) & (df["price"] <= upper)]

# Check dataset size after removing outliers
print(df.shape)

(73100, 17)


In [23]:
df.to_csv("clean_dynamic_pricing_dataset.csv", index=False)